In [ ]:
# Upload API Key

from google.colab import files
files.upload()  # select your kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"meetnikishparikh","key":"9318951fe77ed101009af907bf6ba5fb"}'}

In [ ]:
# Move API Key

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
# Download dataset

!kaggle datasets download -d nguyenhung1903/nerf-synthetic-dataset



Dataset URL: https://www.kaggle.com/datasets/nguyenhung1903/nerf-synthetic-dataset
License(s): unknown
 98% 1.16G/1.18G [00:13<00:00, 255MB/s]
100% 1.18G/1.18G [00:13<00:00, 94.0MB/s]


In [ ]:
# Unzip dataset

!unzip nerf-synthetic-dataset.zip -d ./data/



Archive:  nerf-synthetic-dataset.zip
  inflating: ./data/nerf_synthetic/README.txt  
  inflating: ./data/nerf_synthetic/chair/test/r_0.png  
  inflating: ./data/nerf_synthetic/chair/test/r_0_depth_0000.png  
  inflating: ./data/nerf_synthetic/chair/test/r_1.png  
  inflating: ./data/nerf_synthetic/chair/test/r_10.png  
  inflating: ./data/nerf_synthetic/chair/test/r_100.png  
  inflating: ./data/nerf_synthetic/chair/test/r_100_depth_0000.png  
  inflating: ./data/nerf_synthetic/chair/test/r_101.png  
  inflating: ./data/nerf_synthetic/chair/test/r_101_depth_0000.png  
  inflating: ./data/nerf_synthetic/chair/test/r_102.png  
  inflating: ./data/nerf_synthetic/chair/test/r_102_depth_0000.png  
  inflating: ./data/nerf_synthetic/chair/test/r_103.png  
  inflating: ./data/nerf_synthetic/chair/test/r_103_depth_0000.png  
  inflating: ./data/nerf_synthetic/chair/test/r_104.png  
  inflating: ./data/nerf_synthetic/chair/test/r_104_depth_0000.png  
  inflating: ./data/nerf_synthetic/chair/tes

In [ ]:
# %cd /content
# !mv data/nerf_synthetic/nerf_synthetic /data/nerf_synthetic/
# # !rm -r data/nerf_synthetic/nerf_synthetic


/content
mv: cannot move 'data/nerf_synthetic/nerf_synthetic' to '/data/nerf_synthetic/': No such file or directory


In [ ]:
!nvidia-smi -L
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install kaggle

GPU 0: Tesla T4 (UUID: GPU-9318960a-78ea-6abb-debb-f38c1d27c261)
Mon Sep  8 19:12:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |       

In [ ]:
!pip install tqdm scikit-image opencv-python configargparse lpips imageio-ffmpeg kornia tensorboard plyfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 39.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lpips]


In [ ]:
import torch

ckpt_path = './logs/lego/checkpoints/lego.th'

try:
    ckpt = torch.load(ckpt_path, weights_only=False)
    print("Checkpoint loaded successfully!")
except Exception as e:
    print("Failed to load checkpoint:", e)


Failed to load checkpoint: [Errno 2] No such file or directory: './logs/lego/checkpoints/lego.th'


In [ ]:
%cd /content/TensoRF
!mkdir logs
!mkdir logs/lego
!mkdir logs/lego/checkpoints


/content/TensoRF


In [ ]:
%cd /content/TensoRF


/content/TensoRF


In [ ]:

import os
from tqdm.auto import tqdm
from opt import config_parser



import json, random
from renderer import *
from utils import *
from torch.utils.tensorboard import SummaryWriter
import datetime

from dataLoader import dataset_dict
import sys

sys.argv = [
    "train.py",
    "--config", "configs/lego.txt",
    "--ckpt", "./logs/lego/checkpoints/lego.th",
    "--render_only", "1",
    "--render_test", "1",
    "--datadir", "../data/nerf_synthetic/lego"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

renderer = OctreeRender_trilinear_fast


class SimpleSampler:
    def __init__(self, total, batch):
        self.total = total
        self.batch = batch
        self.curr = total
        self.ids = None

    def nextids(self):
        self.curr+=self.batch
        if self.curr + self.batch > self.total:
            self.ids = torch.LongTensor(np.random.permutation(self.total))
            self.curr = 0
        return self.ids[self.curr:self.curr+self.batch]


@torch.no_grad()
def export_mesh(args):

    ckpt = torch.load(args.ckpt, map_location=device,weights_only=False)
    kwargs = ckpt['kwargs']
    kwargs.update({'device': device})
    tensorf = eval(args.model_name)(**kwargs)
    tensorf.load(ckpt)

    alpha,_ = tensorf.getDenseAlpha()
    convert_sdf_samples_to_ply(alpha.cpu(), f'{args.ckpt[:-3]}.ply',bbox=tensorf.aabb.cpu(), level=0.005)


@torch.no_grad()
def render_test(args):
    # init dataset
    dataset = dataset_dict[args.dataset_name]
    test_dataset = dataset(args.datadir, split='test', downsample=args.downsample_train, is_stack=True)
    white_bg = test_dataset.white_bg
    ndc_ray = args.ndc_ray

    if not os.path.exists(args.ckpt):
        print('the ckpt path does not exists!!')
        return

    ckpt = torch.load(args.ckpt, map_location=device,weights_only=False)
    kwargs = ckpt['kwargs']
    kwargs.update({'device': device})
    tensorf = eval(args.model_name)(**kwargs)
    tensorf.load(ckpt)

    logfolder = os.path.dirname(args.ckpt)
    if args.render_train:
        os.makedirs(f'{logfolder}/imgs_train_all', exist_ok=True)
        train_dataset = dataset(args.datadir, split='train', downsample=args.downsample_train, is_stack=True)
        PSNRs_test = evaluation(train_dataset,tensorf, args, renderer, f'{logfolder}/imgs_train_all/',
                                N_vis=-1, N_samples=-1, white_bg = white_bg, ndc_ray=ndc_ray,device=device)
        print(f'======> {args.expname} train all psnr: {np.mean(PSNRs_test)} <========================')

    if args.render_test:
        os.makedirs(f'{logfolder}/{args.expname}/imgs_test_all', exist_ok=True)
        evaluation(test_dataset,tensorf, args, renderer, f'{logfolder}/{args.expname}/imgs_test_all/',
                                N_vis=-1, N_samples=-1, white_bg = white_bg, ndc_ray=ndc_ray,device=device)

    if args.render_path:
        c2ws = test_dataset.render_path
        os.makedirs(f'{logfolder}/{args.expname}/imgs_path_all', exist_ok=True)
        evaluation_path(test_dataset,tensorf, c2ws, renderer, f'{logfolder}/{args.expname}/imgs_path_all/',
                                N_vis=-1, N_samples=-1, white_bg = white_bg, ndc_ray=ndc_ray,device=device)

def reconstruction(args):

    # init dataset
    dataset = dataset_dict[args.dataset_name]
    train_dataset = dataset(args.datadir, split='train', downsample=args.downsample_train, is_stack=False)
    test_dataset = dataset(args.datadir, split='test', downsample=args.downsample_train, is_stack=True)
    white_bg = train_dataset.white_bg
    near_far = train_dataset.near_far
    ndc_ray = args.ndc_ray

    # init resolution
    upsamp_list = args.upsamp_list
    update_AlphaMask_list = args.update_AlphaMask_list
    n_lamb_sigma = args.n_lamb_sigma
    n_lamb_sh = args.n_lamb_sh


    if args.add_timestamp:
        logfolder = f'{args.basedir}/{args.expname}{datetime.datetime.now().strftime("-%Y%m%d-%H%M%S")}'
    else:
        logfolder = f'{args.basedir}/{args.expname}'


    # init log file
    os.makedirs(logfolder, exist_ok=True)
    os.makedirs(f'{logfolder}/imgs_vis', exist_ok=True)
    os.makedirs(f'{logfolder}/imgs_rgba', exist_ok=True)
    os.makedirs(f'{logfolder}/rgba', exist_ok=True)
    summary_writer = SummaryWriter(logfolder)



    # init parameters
    # tensorVM, renderer = init_parameters(args, train_dataset.scene_bbox.to(device), reso_list[0])
    aabb = train_dataset.scene_bbox.to(device)
    reso_cur = N_to_reso(args.N_voxel_init, aabb)
    nSamples = min(args.nSamples, cal_n_samples(reso_cur,args.step_ratio))


    if args.ckpt is not None:
        ckpt = torch.load(args.ckpt, map_location=device,weights_only=False)
        kwargs = ckpt['kwargs']
        kwargs.update({'device':device})
        tensorf = eval(args.model_name)(**kwargs)
        tensorf.load(ckpt)
    else:
        tensorf = eval(args.model_name)(aabb, reso_cur, device,
                    density_n_comp=n_lamb_sigma, appearance_n_comp=n_lamb_sh, app_dim=args.data_dim_color, near_far=near_far,
                    shadingMode=args.shadingMode, alphaMask_thres=args.alpha_mask_thre, density_shift=args.density_shift, distance_scale=args.distance_scale,
                    pos_pe=args.pos_pe, view_pe=args.view_pe, fea_pe=args.fea_pe, featureC=args.featureC, step_ratio=args.step_ratio, fea2denseAct=args.fea2denseAct)


    grad_vars = tensorf.get_optparam_groups(args.lr_init, args.lr_basis)
    if args.lr_decay_iters > 0:
        lr_factor = args.lr_decay_target_ratio**(1/args.lr_decay_iters)
    else:
        args.lr_decay_iters = args.n_iters
        lr_factor = args.lr_decay_target_ratio**(1/args.n_iters)

    print("lr decay", args.lr_decay_target_ratio, args.lr_decay_iters)

    optimizer = torch.optim.Adam(grad_vars, betas=(0.9,0.99))


    #linear in logrithmic space
    N_voxel_list = (torch.round(torch.exp(torch.linspace(np.log(args.N_voxel_init), np.log(args.N_voxel_final), len(upsamp_list)+1))).long()).tolist()[1:]


    torch.cuda.empty_cache()
    PSNRs,PSNRs_test = [],[0]

    allrays, allrgbs = train_dataset.all_rays, train_dataset.all_rgbs
    if not args.ndc_ray:
        allrays, allrgbs = tensorf.filtering_rays(allrays, allrgbs, bbox_only=True)
    trainingSampler = SimpleSampler(allrays.shape[0], args.batch_size)

    Ortho_reg_weight = args.Ortho_weight
    print("initial Ortho_reg_weight", Ortho_reg_weight)

    L1_reg_weight = args.L1_weight_inital
    print("initial L1_reg_weight", L1_reg_weight)
    TV_weight_density, TV_weight_app = args.TV_weight_density, args.TV_weight_app
    tvreg = TVLoss()
    print(f"initial TV_weight density: {TV_weight_density} appearance: {TV_weight_app}")


    pbar = tqdm(range(args.n_iters), miniters=args.progress_refresh_rate, file=sys.stdout)
    for iteration in pbar:


        ray_idx = trainingSampler.nextids()
        rays_train, rgb_train = allrays[ray_idx], allrgbs[ray_idx].to(device)

        #rgb_map, alphas_map, depth_map, weights, uncertainty
        rgb_map, alphas_map, depth_map, weights, uncertainty = renderer(rays_train, tensorf, chunk=args.batch_size,
                                N_samples=nSamples, white_bg = white_bg, ndc_ray=ndc_ray, device=device, is_train=True)

        loss = torch.mean((rgb_map - rgb_train) ** 2)


        # loss
        total_loss = loss
        if Ortho_reg_weight > 0:
            loss_reg = tensorf.vector_comp_diffs()
            total_loss += Ortho_reg_weight*loss_reg
            summary_writer.add_scalar('train/reg', loss_reg.detach().item(), global_step=iteration)
        if L1_reg_weight > 0:
            loss_reg_L1 = tensorf.density_L1()
            total_loss += L1_reg_weight*loss_reg_L1
            summary_writer.add_scalar('train/reg_l1', loss_reg_L1.detach().item(), global_step=iteration)

        if TV_weight_density>0:
            TV_weight_density *= lr_factor
            loss_tv = tensorf.TV_loss_density(tvreg) * TV_weight_density
            total_loss = total_loss + loss_tv
            summary_writer.add_scalar('train/reg_tv_density', loss_tv.detach().item(), global_step=iteration)
        if TV_weight_app>0:
            TV_weight_app *= lr_factor
            loss_tv = tensorf.TV_loss_app(tvreg)*TV_weight_app
            total_loss = total_loss + loss_tv
            summary_writer.add_scalar('train/reg_tv_app', loss_tv.detach().item(), global_step=iteration)

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        loss = loss.detach().item()

        PSNRs.append(-10.0 * np.log(loss) / np.log(10.0))
        summary_writer.add_scalar('train/PSNR', PSNRs[-1], global_step=iteration)
        summary_writer.add_scalar('train/mse', loss, global_step=iteration)


        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * lr_factor

        # Print the current values of the losses.
        if iteration % args.progress_refresh_rate == 0:
            pbar.set_description(
                f'Iteration {iteration:05d}:'
                + f' train_psnr = {float(np.mean(PSNRs)):.2f}'
                + f' test_psnr = {float(np.mean(PSNRs_test)):.2f}'
                + f' mse = {loss:.6f}'
            )
            PSNRs = []


        if iteration % args.vis_every == args.vis_every - 1 and args.N_vis!=0:
            PSNRs_test = evaluation(test_dataset,tensorf, args, renderer, f'{logfolder}/imgs_vis/', N_vis=args.N_vis,
                                    prtx=f'{iteration:06d}_', N_samples=nSamples, white_bg = white_bg, ndc_ray=ndc_ray, compute_extra_metrics=False)
            summary_writer.add_scalar('test/psnr', np.mean(PSNRs_test), global_step=iteration)



        if iteration in update_AlphaMask_list:

            if reso_cur[0] * reso_cur[1] * reso_cur[2]<256**3:# update volume resolution
                reso_mask = reso_cur
            new_aabb = tensorf.updateAlphaMask(tuple(reso_mask))
            if iteration == update_AlphaMask_list[0]:
                tensorf.shrink(new_aabb)
                # tensorVM.alphaMask = None
                L1_reg_weight = args.L1_weight_rest
                print("continuing L1_reg_weight", L1_reg_weight)


            if not args.ndc_ray and iteration == update_AlphaMask_list[1]:
                # filter rays outside the bbox
                allrays,allrgbs = tensorf.filtering_rays(allrays,allrgbs)
                trainingSampler = SimpleSampler(allrgbs.shape[0], args.batch_size)


        if iteration in upsamp_list:
            n_voxels = N_voxel_list.pop(0)
            reso_cur = N_to_reso(n_voxels, tensorf.aabb)
            nSamples = min(args.nSamples, cal_n_samples(reso_cur,args.step_ratio))
            tensorf.upsample_volume_grid(reso_cur)

            if args.lr_upsample_reset:
                print("reset lr to initial")
                lr_scale = 1 #0.1 ** (iteration / args.n_iters)
            else:
                lr_scale = args.lr_decay_target_ratio ** (iteration / args.n_iters)
            grad_vars = tensorf.get_optparam_groups(args.lr_init*lr_scale, args.lr_basis*lr_scale)
            optimizer = torch.optim.Adam(grad_vars, betas=(0.9, 0.99))


    tensorf.save(f'{logfolder}/{args.expname}.th')


    if args.render_train:
        os.makedirs(f'{logfolder}/imgs_train_all', exist_ok=True)
        train_dataset = dataset(args.datadir, split='train', downsample=args.downsample_train, is_stack=True)
        PSNRs_test = evaluation(train_dataset,tensorf, args, renderer, f'{logfolder}/imgs_train_all/',
                                N_vis=-1, N_samples=-1, white_bg = white_bg, ndc_ray=ndc_ray,device=device)
        print(f'======> {args.expname} test all psnr: {np.mean(PSNRs_test)} <========================')

    if args.render_test:
        os.makedirs(f'{logfolder}/imgs_test_all', exist_ok=True)
        PSNRs_test = evaluation(test_dataset,tensorf, args, renderer, f'{logfolder}/imgs_test_all/',
                                N_vis=-1, N_samples=-1, white_bg = white_bg, ndc_ray=ndc_ray,device=device)
        summary_writer.add_scalar('test/psnr_all', np.mean(PSNRs_test), global_step=iteration)
        print(f'======> {args.expname} test all psnr: {np.mean(PSNRs_test)} <========================')

    if args.render_path:
        c2ws = test_dataset.render_path
        # c2ws = test_dataset.poses
        print('========>',c2ws.shape)
        os.makedirs(f'{logfolder}/imgs_path_all', exist_ok=True)
        evaluation_path(test_dataset,tensorf, c2ws, renderer, f'{logfolder}/imgs_path_all/',
                                N_vis=-1, N_samples=-1, white_bg = white_bg, ndc_ray=ndc_ray,device=device)


if __name__ == '__main__':

    torch.set_default_dtype(torch.float32)
    torch.manual_seed(20211202)
    np.random.seed(20211202)

    args = config_parser()
    print(args)

    if  args.export_mesh:
        export_mesh(args)

    if args.render_only and (args.render_test or args.render_path):
        render_test(args)
    else:
        reconstruction(args)

Namespace(config='configs/lego.txt', expname='tensorf_lego_VM', basedir='./log', add_timestamp=0, datadir='../data/nerf_synthetic/lego', progress_refresh_rate=10, with_depth=False, downsample_train=1.0, downsample_test=1.0, model_name='TensorCP', batch_size=1024, n_iters=50, dataset_name='blender', lr_init=0.02, lr_basis=0.001, lr_decay_iters=-1, lr_decay_target_ratio=0.1, lr_upsample_reset=1, L1_weight_inital=1e-05, L1_weight_rest=1e-05, Ortho_weight=0.0, TV_weight_density=0.0, TV_weight_app=0.0, n_lamb_sigma=[96], n_lamb_sh=[288], data_dim_color=27, rm_weight_mask_thre=0.0001, alpha_mask_thre=0.0001, distance_scale=25, density_shift=-10, shadingMode='MLP_Fea', pos_pe=6, view_pe=2, fea_pe=2, featureC=128, ckpt='./logs/lego/checkpoints/lego.th', render_only=1, render_test=1, render_train=0, render_path=0, export_mesh=0, lindisp=False, perturb=1.0, accumulate_decay=0.998, fea2denseAct='softplus', ndc_ray=0, nSamples=1000000.0, step_ratio=0.5, white_bkgd=False, N_voxel_init=2097156, N_vo

Loading data test (200): 100%|██████████| 200/200 [00:09<00:00, 20.14it/s]


aabb tensor([-1.5000, -1.5000, -1.5000,  1.5000,  1.5000,  1.4528], device='cuda:0')
grid size [502, 502, 494]
sampling step size:  tensor(0.0030, device='cuda:0')
sampling number:  1727
pos_pe 6 view_pe 2 fea_pe 2
MLPRender_Fea(
  (mlp): Sequential(
    (0): Linear(in_features=150, out_features=128, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=128, out_features=3, bias=True)
  )
)


0it [00:00, ?it/s]

init_lpips: lpips_alex
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


...

Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
init_lpips: lpips_vgg
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth



  0%|          | 0.00/528M [00:00<?, ?B/s]
  2%|▏         | 13.0M/528M [00:00<00:03, 136MB/s]
  6%|▌         | 29.1M/528M [00:00<00:03, 155MB/s]
  8%|▊         | 44.0M/528M [00:01<00:24, 20.4MB/s]
 11%|█▏        | 60.2M/528M [00:01<00:15, 32.2MB/s]
 15%|█▍        | 78.4M/528M [00:01<00:09, 48.2MB/s]
 18%|█▊        | 95.4M/528M [00:02<00:07, 64.6MB/s]
 21%|██        | 110M/528M [00:02<00:05, 74.7MB/s] 
 24%|██▎       | 124M/528M [00:02<00:05, 83.3MB/s]
 26%|██▌       | 137M/528M [00:02<00:04, 88.8MB/s]
 28%|██▊       | 150M/528M [00:02<00:04, 97.6MB/s]
 31%|███       | 162M/528M [00:02<00:03, 104MB/s] 
 33%|███▎      | 175M/528M [00:02<00:03, 112MB/s]
 35%|███▌      | 187M/528M [00:02<00:03, 114MB/s]
 38%|███▊      | 199M/528M [00:02<00:02, 116MB/s]
 40%|████      | 212M/528M [00:03<00:02, 119MB/s]
 42%|████▏     | 224M/528M [00:03<00:02, 121MB/s]
 45%|████▍     | 236M/528M [00:03<00:02, 117MB/s]
 47%|████▋     | 248M/528M [00:03<00:02, 117MB/s]
 49%|████▉     | 260M/528M [00:03<00:02,

Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth


In [ ]:
# ZIP Results to export

!cd /content/TensoRF
!zip -r /content/results.zip ./logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all

  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/ (stored 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/025.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/027.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/088.png (deflated 1%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/142.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/071.png (deflated 1%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/mean.txt (deflated 28%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/144.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/033.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/026.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/001.png (deflated 0%)
  adding: logs/lego/checkpoints/tensorf_lego_VM/imgs_test_all/148.png (deflated 1%)

In [ ]:
# Download results

from google.colab import files
files.download('/content/results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>